# SAMPLE-DATASET-1 quickstart

This notebook reads the two Parquet files in SAMPLE-DATASET-1. `facts.parquet` has one row per numeric fact in the /v1/facts row shape, for every filing of the 50 selected companies, with no as_of cutoff. `revisions.parquet` has one row per revision event: the seven /v1/revisions event fields plus five series-identity columns added by the export. The next cell loads both files and prints their shapes.

In [ ]:
# Fetch the two Parquet files when they are not next to the notebook (for Colab).
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/jushuea/arkleon-sample/main"

for name in ("facts.parquet", "revisions.parquet"):
    if not os.path.exists(name):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", name)
        print(f"Downloaded {name}")


In [ ]:
import pandas as pd

facts = pd.read_parquet("facts.parquet", engine="pyarrow")
revisions = pd.read_parquet("revisions.parquet", engine="pyarrow")
print("facts shape:", facts.shape)
print("revisions shape:", revisions.shape)


## Columns

The next cell prints the dtypes of the facts table and its first five rows, sorted by `accession_id` and `tag`. `field_dictionary.md`, next to this notebook, gives every column's Arrow type, meaning, and contract section. Two columns carry the point-in-time logic used below. `filed` is the filing date. `period_end` is the end of the period a fact describes and takes no part in an as_of cut.

In [ ]:
print(facts.dtypes)
facts.sort_values(["accession_id", "tag"]).head(5)


## Example series

The next cell filters `facts` to one series: CIK 831001 (CITIGROUP INC), tag `EarningsPerShareDiluted`, `period_end` 2009-12-31, `duration_quarters` 4, `unit` USD, consolidated (`segments` and `coreg` both empty strings). Each remaining row is that fact as one filing reported it, with the filing's `filed` date and `accession_id`.

In [ ]:
CIK = 831001
TAG = 'EarningsPerShareDiluted'
PERIOD_END = pd.Timestamp('2009-12-31').date()
QTRS = 4
UNIT = 'USD'

series = facts.loc[
    facts["cik"].eq(CIK)
    & facts["tag"].eq(TAG)
    & pd.to_datetime(facts["period_end"]).dt.date.eq(PERIOD_END)
    & facts["duration_quarters"].eq(QTRS)
    & facts["unit"].eq(UNIT)
    & facts["segments"].eq("")
    & facts["coreg"].eq("")
    & facts["value"].notna()
].copy()
series["filed"] = pd.to_datetime(series["filed"]).dt.date
series = series.sort_values(["filed", "accession_id"])
series[["accession_id", "form", "filed", "value", "taxonomy"]]


## First filed versus later reported

The series has more than one filing. The two the rest of this notebook uses:

| Filing | `accession_id` | `filed` | `value` |
|---|---|---|---|
| First filed | 0001206774-10-000406 | 2010-02-26 | -0.8 |
| Later reported | 0001206774-12-000799 | 2012-02-24 | -7.99 |

Both rows describe the same period, ending 2009-12-31. They differ in `filed` and in `value`. The next cell prints the two rows.

In [ ]:
assert not series.empty, "The example series must contain a first filing."
first = series.iloc[0]
changed = series.loc[series["value"].ne(first["value"])]
assert not changed.empty, "The example series must contain a different later value."
latest = changed.iloc[-1]
assert first is not None and latest is not None
assert first["value"] != latest["value"]
print(f"first filed {first['filed']} in {first['accession_id']}: {first['value']}")
print(f"later reported {latest['filed']} in {latest['accession_id']}: {latest['value']}")


## The as_of rule, applied locally

The API contract (section 2.1.1) states that `/v1/facts` returns only facts whose filing was filed on or before `as_of`. The filter is on the filing date, never on the period. These files carry every filing with no `as_of` cutoff, so the next cell applies the rule itself.

`value_as_of` keeps the rows of the series whose `filed` date is on or before `as_of` and returns the value from the latest of those filings. The cell prints it for two dates:

- `as_of` 2011-01-01: only the 2010-02-26 filing is visible. The function returns -0.8.
- `as_of` 2013-01-01: the 2012-02-24 filing is also visible. The function returns -7.99.

In [ ]:
def value_as_of(df, as_of):
    cutoff = pd.Timestamp(as_of).date()
    eligible = df.copy()
    eligible["filed"] = pd.to_datetime(eligible["filed"]).dt.date
    eligible = eligible.loc[eligible["filed"] <= cutoff]
    if eligible.empty:
        return None
    return eligible.sort_values(["filed", "accession_id"]).iloc[-1]["value"]

for as_of in ("2011-01-01", "2013-01-01"):
    print(f"as_of {as_of}: {value_as_of(series, as_of)}")


## Matching revision events

The next cell selects the rows of `revisions.parquet` with the same series key: `cik` 831001, `tag` `EarningsPerShareDiluted`, `period_end` 2009-12-31, `duration_quarters` 4, `unit` USD. The event for accession 0001206774-12-000799, filed 2012-02-24, carries `val` -7.99 and `event_type` `RESTATED_COMPARATIVE`.

The lineage columns:

- `event_type` is one of `AMENDMENT` or `RESTATED_COMPARATIVE`. It is an event-type identifier on the corpus row, not a characterization of the filing.
- `supersedes_accn` is the accession that this event's filing superseded, or null.
- `superseded_by_accn` is the accession that later superseded this event's filing, or null.

The five identity columns come from the sample export. The other seven columns follow the `/v1/revisions` response.

In [ ]:
matching_revisions = revisions.loc[
    revisions["cik"].eq(CIK)
    & revisions["tag"].eq(TAG)
    & pd.to_datetime(revisions["period_end"]).dt.date.eq(PERIOD_END)
    & revisions["duration_quarters"].eq(QTRS)
    & revisions["unit"].eq(UNIT)
].copy()
matching_revisions["filed"] = pd.to_datetime(matching_revisions["filed"]).dt.date
matching_revisions.sort_values(["filed", "accession_id"])[
    ["val", "filed", "accession_id", "form", "event_type",
     "supersedes_accn", "superseded_by_accn"]
]
